# Introduction to Feature Engineering

Feature engineering is the process of using domain knowledge to create new input variables (features) from raw data, or to transform existing ones, so that machine learning models can find patterns more easily. While feature scaling changes the range of a feature, feature engineering changes what information is even available to the model in the first place.

**Why It Matters**

•	A well-engineered feature can outperform a more complex model trained on raw data alone.

•	Raw columns often hide useful signal — e.g. Kilometer alone says little, but Kilometer per Year of ownership says a lot about usage intensity.

•	Good features reduce the burden on the model to "discover" relationships it may never find on its own.

•	It is often said: "garbage in, garbage out" — the quality of features usually matters more than the choice of algorithm.


•	A well-engineered feature can outperform a more complex model trained on raw data alone.

•	Raw columns often hide useful signal — e.g. Kilometer alone says little, but Kilometer per Year of ownership says a lot about usage intensity.

•	Good features reduce the burden on the model to "discover" relationships it may never find on its own.

•	It is often said: "garbage in, garbage out" — the quality of features usually matters more than the choice of algorithm.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("used_cars_dataset.csv")
print(df.columns.tolist())
print(df.head())


**Creating New Features**

Price per Kilometer Driven

In [ ]:
df['Price_per_Km'] = df['Price'] / (df['Kilometer'] + 1)  # +1 avoids divide-by-zero

Captures value relative to usage — a cheap car with very high mileage may actually be a worse deal than it first appears

Kilometers Driven per Year

In [ ]:
df['Km_per_Year'] = df['Kilometer'] / df['Age_Years'].replace(0, 1)

Normalizes total distance by how long the car has been owned, giving a better sense of usage intensity than raw Kilometer alone.

Power-to-Engine Ratio

A simple efficiency indicator — how much power the engine produces per unit of displacement.


In [ ]:
df['Power_per_CC'] = df['Power_BHP'] / df['Engine_CC']

Is Luxury Brand Flag

In [ ]:
luxury_brands = ['Mercedes', 'BMW', 'Audi']
df['Is_Luxury_Brand'] = df['Brand'].isin(luxury_brands).astype(int)


**Date & Time Feature Extraction**

Date and time columns are rarely useful to a model in raw form — a timestamp like "2023-05-14" carries no numeric meaning by itself. Instead, we extract structured components: Year, Month, Day, and Day of Week.


Extracting Year
  




In [ ]:
df['Listing_Year'] = df['Listing_Date'].dt.year

Extracting Month



In [ ]:
df['Listing_Month'] = df['Listing_Date'].dt.month

Extracting Day



In [ ]:
df['Listing_Day'] = df['Listing_Date'].dt.day

Extracting Day of Week

Day of week can reveal patterns — e.g. more listings posted on weekends. dayofweek returns 0 (Monday) through 6 (Sunday); day_name() gives a readable label.


In [ ]:
df['Listing_DayOfWeek'] = df['Listing_Date'].dt.dayofweek
df['Listing_DayName'] = df['Listing_Date'].dt.day_name()

print(df[['Listing_Date', 'Listing_Year', 'Listing_Month',
          'Listing_Day', 'Listing_DayOfWeek', 'Listing_DayName']].head())


**Key Takeaway**
Once split into Year, Month, Day, and Day of Week, a date column becomes several numeric/categorical features a model can actually use — for example, detecting seasonal pricing patterns.


**Binning / Bucketing**

Binning converts a continuous numeric feature into a smaller number of discrete categories (bins). This can reduce noise, handle outliers, and make relationships easier for simpler models to capture.


In [ ]:
df['Age_Bin'] = pd.cut(
    df['Age_Years'],
    bins=[0, 3, 7, 12, 25],
    labels=['New (0-3y)', 'Recent (4-7y)', 'Used (8-12y)', 'Old (13y+)']
)


**Text-Based Feature Extraction**

Text columns like Model or Brand contain structured signal hidden inside free text — length, word count, and specific keywords can all become useful numeric or binary features.

**Text Length and Word Count**


In [ ]:
df['Model_Name_Length'] = df['Model'].astype(str).apply(len)
df['Model_Word_Count'] = df['Model'].astype(str).apply(lambda x: len(x.split()))
